In [0]:
landing_table=dbutils.widgets.get("landing_table")
landing_flattened_table=dbutils.widgets.get("landing_flattened_table")
raw_table=dbutils.widgets.get("raw_table")
volume_path=dbutils.widgets.get("volume_path")

In [0]:
try:
    spark.sql(f"""
        WITH latest_src AS (
            SELECT *
            FROM (
                SELECT
                    *,
                    ROW_NUMBER() OVER (
                        PARTITION BY TransID, ProvAdjustments_AdjAmount, ProvAdjustments_PeriodDate, ProvAdjustments_ProviderID, ProvAdjustments_ReasonCode, ProvAdjustments_ReferenceID
                        ORDER BY _load_timestamp DESC
                    ) AS rn
                FROM {landing_flattened_table}
            ) t
            WHERE rn = 1
        )

        MERGE INTO {raw_table} tgt
        USING latest_src src
        ON tgt.TransID = src.TransID
        AND COALESCE(tgt.ProvAdjustments_AdjAmount, CAST(0 AS DECIMAL(10,2))) = COALESCE(src.ProvAdjustments_AdjAmount, CAST(0 AS DECIMAL(10,2)))
        AND COALESCE(tgt.ProvAdjustments_PeriodDate,'') = COALESCE(src.ProvAdjustments_PeriodDate,'')
        AND COALESCE(tgt.ProvAdjustments_ProviderID,0) = COALESCE(src.ProvAdjustments_ProviderID,0)
        AND COALESCE(tgt.ProvAdjustments_ReasonCode,'') = COALESCE(src.ProvAdjustments_ReasonCode,'')
        AND COALESCE(tgt.ProvAdjustments_ReferenceID,'') = COALESCE(src.ProvAdjustments_ReferenceID,'')

        AND tgt._modified_ts < src._load_timestamp
        AND tgt._active_flag = 1

        WHEN MATCHED THEN
            UPDATE SET
                tgt._active_flag = 0,
                tgt._modified_ts = src._load_timestamp
    """)
except Exception as e:
    print(f"Error updating {raw_table}: {e}")
    raise

In [0]:
try:
    spark.sql(f"""
        WITH latest_src AS (
            SELECT *
            FROM (
                SELECT
                    *,
                    ROW_NUMBER() OVER (
                        PARTITION BY TransID, ProvAdjustments_AdjAmount, ProvAdjustments_PeriodDate, ProvAdjustments_ProviderID, ProvAdjustments_ReasonCode, ProvAdjustments_ReferenceID
                        ORDER BY _load_timestamp DESC
                    ) AS rn
                FROM {landing_flattened_table}
            ) t
            WHERE rn = 1
        )

        MERGE INTO {raw_table} tgt
        USING latest_src src
        ON tgt.TransID = src.TransID
        AND COALESCE(tgt.ProvAdjustments_AdjAmount, CAST(0 AS DECIMAL(10,2))) = COALESCE(src.ProvAdjustments_AdjAmount, CAST(0 AS DECIMAL(10,2)))
        AND COALESCE(tgt.ProvAdjustments_PeriodDate,'') = COALESCE(src.ProvAdjustments_PeriodDate,'')
        AND COALESCE(tgt.ProvAdjustments_ProviderID,0) = COALESCE(src.ProvAdjustments_ProviderID,0)
        AND COALESCE(tgt.ProvAdjustments_ReasonCode,'') = COALESCE(src.ProvAdjustments_ReasonCode,'')
        AND COALESCE(tgt.ProvAdjustments_ReferenceID,'') = COALESCE(src.ProvAdjustments_ReferenceID,'')

        AND tgt._created_ts = src._load_timestamp
        
        WHEN NOT MATCHED THEN
          INSERT (
              AssignedToID,
              CheckNumber,
              ClaimType,
              CreateDate,
              CreateMode,
              CreditDebitFlag,
              CustomGroup,
              DepositIndicator,
              DownloadIndicator,
              EditByID,
              EditDate,
              ImportBatchID,
              LastUpdated,
              LastUpdatedUserId,
              LastUserUpdatedDate,
              MediaCode,
              OrigCompanyID,
              ParentID,
              PatientID,
              PayDate,
              PayFormat,
              PayMethod,
              PayeeAddr1,
              PayeeAddr2,
              PayeeCity,
              PayeeID_EIN,
              PayeeID_NPI,
              PayeeID_PayeeID,
              PayeeID_TaxID,
              PayeeName,
              PayeeState,
              PayeeZip,
              PayerAddr1,
              PayerAddr2,
              PayerCity,
              PayerContEmail,
              PayerContExt,
              PayerContName,
              PayerContPhone,
              PayerID,
              PayerMatchID,
              PayerName,
              PayerPartnerID,
              PayerState,
              PayerZip,
              PaymentAmt,
              PostedDate,
              PostedIndicator,
              ProductionDate,
              ProvAdjustments_AdjAmount,
              ProvAdjustments_PeriodDate,
              ProvAdjustments_ProviderID,
              ProvAdjustments_ReasonCode,
              ProvAdjustments_ReferenceID,
              ProvMatchID,
              ProvPartnerID,
              ReceiverAcctNo,
              ReceiverDFINo,
              ReceiverID,
              SenderAcctNo,
              SenderDFINo,
              SourceFileID,
              TransHandleCode,
              TransID,
              TransStatus,
              TransType,
              TransmitDate,
              VersionID,
              _file_name,
              _created_ts,
              _modified_ts,
              _active_flag
          )
          VALUES (
              src.AssignedToID,
              src.CheckNumber,
              src.ClaimType,
              src.CreateDate,
              src.CreateMode,
              src.CreditDebitFlag,
              src.CustomGroup,
              src.DepositIndicator,
              src.DownloadIndicator,
              src.EditByID,
              src.EditDate,
              src.ImportBatchID,
              src.LastUpdated,
              src.LastUpdatedUserId,
              src.LastUserUpdatedDate,
              src.MediaCode,
              src.OrigCompanyID,
              src.ParentID,
              src.PatientID,
              src.PayDate,
              src.PayFormat,
              src.PayMethod,
              src.PayeeAddr1,
              src.PayeeAddr2,
              src.PayeeCity,
              src.PayeeID_EIN,
              src.PayeeID_NPI,
              src.PayeeID_PayeeID,
              src.PayeeID_TaxID,
              src.PayeeName,
              src.PayeeState,
              src.PayeeZip,
              src.PayerAddr1,
              src.PayerAddr2,
              src.PayerCity,
              src.PayerContEmail,
              src.PayerContExt,
              src.PayerContName,
              src.PayerContPhone,
              src.PayerID,
              src.PayerMatchID,
              src.PayerName,
              src.PayerPartnerID,
              src.PayerState,
              src.PayerZip,
              src.PaymentAmt,
              src.PostedDate,
              src.PostedIndicator,
              src.ProductionDate,
              src.ProvAdjustments_AdjAmount,
              src.ProvAdjustments_PeriodDate,
              src.ProvAdjustments_ProviderID,
              src.ProvAdjustments_ReasonCode,
              src.ProvAdjustments_ReferenceID,
              src.ProvMatchID,
              src.ProvPartnerID,
              src.ReceiverAcctNo,
              src.ReceiverDFINo,
              src.ReceiverID,
              src.SenderAcctNo,
              src.SenderDFINo,
              src.SourceFileID,
              src.TransHandleCode,
              src.TransID,
              src.TransStatus,
              src.TransType,
              src.TransmitDate,
              src.VersionID,
              src._file_name,
              src._load_timestamp,
              src._load_timestamp,
              1
          );
    """)
except Exception as e:
    print(f"Error inserting into {raw_table}: {e}")
    raise
